# 33 -- Section 01e Step 3: hidden activation as a hyperbolic probability

Computes q+ = sigmoid(x+) and q- = sigmoid(x-) for the Rumble in the Bronx x h1
worked example, reusing the exact x+/x- values saved by notebook 32, for both
the 01c (b = 0) and 01e (b = -1) scenarios. Also decomposes q = q+ e+ + q- e-
into its real part (q+ + q-)/2 and k part (q+ - q-)/2. Does not touch the HTML.

## Part 0 -- Setup: scan notebooks/outputs, pick this notebook's number

In [1]:
from pathlib import Path
import json
import re
import math

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
notebooks_dir = root / "notebooks"
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

nb_paths = sorted(p for p in notebooks_dir.glob("*.ipynb") if "step3_hidden_probability" not in p.stem)
num_re = re.compile(r"^(\d+)")
numbered = sorted(
    (int(m.group(1)), p) for p in nb_paths if (m := num_re.match(p.stem))
)
highest_num, highest_path = numbered[-1]
this_num = highest_num + 1
this_name = f"{this_num}_step3_hidden_probability.ipynb"
print(f"highest existing notebook: {highest_num:>3d}  ({highest_path.name})")
print(f"this notebook is:          {this_num:>3d}  ({this_name})")

highest existing notebook:  32  (32_step2_hidden_input.ipynb)
this notebook is:           33  (33_step3_hidden_probability.ipynb)


## Part 1 -- Reuse notebook 32's x+ and x- values

In [2]:
results32_path = out_dir / "32_step2_hidden_input_results.json"
assert results32_path.exists(), f"Missing {results32_path}"
results32 = json.loads(results32_path.read_text())

x_plus = results32["x_plus"]
x_minus = results32["x_minus"]
print(f"x+ : 01c={x_plus['01c']:.6f}  01e={x_plus['01e']:.6f}")
print(f"x- : 01c={x_minus['01c']:.6f}  01e={x_minus['01e']:.6f}")

x+ : 01c=-0.211115  01e=-0.217548
x- : 01c=-0.205367  01e=-0.214729


## Part 2 -- q+ = sigmoid(x+), q- = sigmoid(x-), for both scenarios

In [3]:
def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

q_plus = {k: sigmoid(v) for k, v in x_plus.items()}
q_minus = {k: sigmoid(v) for k, v in x_minus.items()}
print(f"q+ : 01c={q_plus['01c']:.6f}  01e={q_plus['01e']:.6f}")
print(f"q- : 01c={q_minus['01c']:.6f}  01e={q_minus['01e']:.6f}")

q+ : 01c=0.447416  01e=0.445826
q- : 01c=0.448838  01e=0.446523


## Part 3 -- Decompose q = q+ e+ + q- e- into real part and k part

`e+ = (1+k)/2`, `e- = (1-k)/2` (defined in section 01d), so the real part of
`q` is `(q+ + q-)/2` and the k part is `(q+ - q-)/2`.

In [4]:
q_real = {k: (q_plus[k] + q_minus[k]) / 2 for k in ("01c", "01e")}
q_k = {k: (q_plus[k] - q_minus[k]) / 2 for k in ("01c", "01e")}
print(f"Re(q) : 01c={q_real['01c']:.6f}  01e={q_real['01e']:.6f}")
print(f"k(q)  : 01c={q_k['01c']:.6f}  01e={q_k['01e']:.6f}")

for scenario in ("01c", "01e"):
    assert 0 < q_plus[scenario] < 1 and 0 < q_minus[scenario] < 1, f"{scenario}: not both in (0,1)"
print("both q+ and q- lie in (0, 1) for both scenarios -- confirmed in code, not assumed.")

Re(q) : 01c=0.448127  01e=0.446175
k(q)  : 01c=-0.000711  01e=-0.000348
both q+ and q- lie in (0, 1) for both scenarios -- confirmed in code, not assumed.


## Part 4 -- Save for Step 3 of section 01e

In [5]:
results = {
    "notebook": this_name,
    "reused_from": "32_step2_hidden_input.ipynb / outputs/32_step2_hidden_input_results.json",
    "movie": results32["movie"],
    "user": results32["user"],
    "hidden_unit": results32["hidden_unit"],
    "x_plus": x_plus,
    "x_minus": x_minus,
    "q_plus": q_plus,
    "q_minus": q_minus,
    "q_real": q_real,
    "q_k": q_k,
}
results_path = out_dir / "33_step3_hidden_probability_results.json"
with results_path.open("w") as f:
    json.dump(results, f, indent=2)
print(f"saved {results_path}")
print(json.dumps(results, indent=2))

saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/33_step3_hidden_probability_results.json
{
  "notebook": "33_step3_hidden_probability.ipynb",
  "reused_from": "32_step2_hidden_input.ipynb / outputs/32_step2_hidden_input_results.json",
  "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
  "user": 24127,
  "hidden_unit": "h1 (index 0)",
  "x_plus": {
    "01c": -0.21111512184143066,
    "01e": -0.21754834055900574
  },
  "x_minus": {
    "01c": -0.20536701381206512,
    "01e": -0.2147289365530014
  },
  "q_plus": {
    "01c": 0.44741637718043453,
    "01e": 0.44582640393415923
  },
  "q_minus": {
    "01c": 0.448837936215285,
    "01e": 0.44652308655726475
  },
  "q_real": {
    "01c": 0.44812715669785974,
    "01e": 0.446174745245712
  },
  "q_k": {
    "01c": -0.0007107795174252363,
    "01e": -0.0003483413115527578
  }
}
